In [1]:
import datasets

ds = datasets.load_dataset("openai/gsm8k", name="main", split="train")

In [2]:
ds

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
        num_rows: 17398
    })
})

In [5]:
ds['train'][0]

{'prompt': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a

In [5]:
# Test DAPO processed environment
from trainer.envs.math_env import (
    DAPOMath17KProcessedDataset,
    DAPOMath17KProcessedEnv,
    extract_think_and_answer,
    DAPO_MATH_SYSTEM_PROMPT
)
from trainer.envs import base_env
import transformers

print("✓ Imports successful")

/home/recoverx/astarag/trainer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/recoverx/astarag/trainer/.venv/lib/python3.13/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✓ Imports successful


In [6]:
# Test 1: Test extract_think_and_answer function
print("=" * 60)
print("Test 1: Testing extract_think_and_answer function")
print("=" * 60)

# Test case 1: Both tags present
test_text1 = """
<think>
This is the reasoning process.
Step 1: Calculate 2 + 2
Step 2: The answer is 4
</think>
<answer>4</answer>
"""
think1, answer1 = extract_think_and_answer(test_text1)
print(f"Test 1.1 - Both tags present:")
print(f"  Think text: {think1}")
print(f"  Answer text: {answer1}")
assert think1 == "This is the reasoning process.\nStep 1: Calculate 2 + 2\nStep 2: The answer is 4", "Think text mismatch"
assert answer1 == "4", "Answer text mismatch"
print("  ✓ PASSED\n")

# Test case 2: Missing think tag
test_text2 = "<answer>5</answer>"
think2, answer2 = extract_think_and_answer(test_text2)
print(f"Test 1.2 - Missing think tag:")
print(f"  Think text: {think2}")
print(f"  Answer text: {answer2}")
assert think2 is None, "Think should be None"
assert answer2 == "5", "Answer text mismatch"
print("  ✓ PASSED\n")

# Test case 3: Missing answer tag
test_text3 = "<think>Some reasoning</think>"
think3, answer3 = extract_think_and_answer(test_text3)
print(f"Test 1.3 - Missing answer tag:")
print(f"  Think text: {think3}")
print(f"  Answer text: {answer3}")
assert think3 == "Some reasoning", "Think text mismatch"
assert answer3 is None, "Answer should be None"
print("  ✓ PASSED\n")

# Test case 4: Both missing
test_text4 = "No tags here"
think4, answer4 = extract_think_and_answer(test_text4)
print(f"Test 1.4 - Both tags missing:")
print(f"  Think text: {think4}")
print(f"  Answer text: {answer4}")
assert think4 is None, "Think should be None"
assert answer4 is None, "Answer should be None"
print("  ✓ PASSED\n")

print("All extract_think_and_answer tests passed! ✓")

Test 1: Testing extract_think_and_answer function
Test 1.1 - Both tags present:
  Think text: This is the reasoning process.
Step 1: Calculate 2 + 2
Step 2: The answer is 4
  Answer text: 4
  ✓ PASSED

Test 1.2 - Missing think tag:
  Think text: None
  Answer text: 5
  ✓ PASSED

Test 1.3 - Missing answer tag:
  Think text: Some reasoning
  Answer text: None
  ✓ PASSED

Test 1.4 - Both tags missing:
  Think text: None
  Answer text: None
  ✓ PASSED

All extract_think_and_answer tests passed! ✓


In [7]:
# Test 2: Test DAPOMath17KProcessedDataset.prepare_dataset
print("=" * 60)
print("Test 2: Testing DAPOMath17KProcessedDataset.prepare_dataset")
print("=" * 60)

dataset_processor = DAPOMath17KProcessedDataset()
ds_name = "open-r1/DAPO-Math-17k-Processed"

print(f"Loading and processing dataset: {ds_name}")
print("This may take a moment due to parallel processing...")

prompts = dataset_processor.prepare_dataset(ds_name)

print(f"\n✓ Dataset loaded successfully!")
print(f"  Total prompts: {len(prompts)}")
print(f"  Type: {type(prompts)}")
print(f"  Expected type: list")

assert isinstance(prompts, list), "prompts should be a list"
assert len(prompts) > 0, "prompts list should not be empty"
print("  ✓ PASSED\n")

# Check structure of first prompt
print("Checking structure of first prompt:")
first_prompt = prompts[0]
print(f"  Type: {type(first_prompt)}")
print(f"  Expected type: {base_env.Prompt}")
assert isinstance(first_prompt, base_env.Prompt), "First item should be a Prompt object"
print("  ✓ PASSED\n")

# Check prompt fields
print("Checking prompt fields:")
print(f"  prompt: {type(first_prompt.prompt)} (length: {len(first_prompt.prompt) if isinstance(first_prompt.prompt, list) else 'N/A'})")
print(f"  data_source: {type(first_prompt.data_source)}")
print(f"  ability: {first_prompt.ability}")
print(f"  reward_model: {type(first_prompt.reward_model)}")
print(f"  extra_info: {type(first_prompt.extra_info)}")

assert isinstance(first_prompt.prompt, list), "prompt should be a list"
assert len(first_prompt.prompt) == 2, "prompt should have 2 messages (system + user)"
assert first_prompt.prompt[0]["role"] == "system", "First message should be system"
assert first_prompt.prompt[1]["role"] == "user", "Second message should be user"
assert first_prompt.ability == "math", "ability should be 'math'"
print("  ✓ PASSED\n")

# Check system prompt
print("Checking system prompt:")
system_content = first_prompt.prompt[0]["content"]
print(f"  System prompt matches expected: {system_content == DAPO_MATH_SYSTEM_PROMPT}")
assert system_content == DAPO_MATH_SYSTEM_PROMPT, "System prompt should match DAPO_MATH_SYSTEM_PROMPT"
print("  ✓ PASSED\n")

# Display sample prompt structure
print("Sample prompt structure:")
print(f"  System message: {first_prompt.prompt[0]['content'][:100]}...")
print(f"  User message: {first_prompt.prompt[1]['content'][:100]}...")
print(f"  Reward model keys: {list(first_prompt.reward_model.keys()) if isinstance(first_prompt.reward_model, dict) else 'N/A'}")

print("\nAll dataset preparation tests passed! ✓")

Test 2: Testing DAPOMath17KProcessedDataset.prepare_dataset
Loading and processing dataset: open-r1/DAPO-Math-17k-Processed
This may take a moment due to parallel processing...


Map: 100%|██████████| 17398/17398 [00:01<00:00, 14223.61 examples/s]



✓ Dataset loaded successfully!
  Total prompts: 17398
  Type: <class 'list'>
  Expected type: list
  ✓ PASSED

Checking structure of first prompt:
  Type: <class 'trainer.envs.base_env.Prompt'>
  Expected type: <class 'trainer.envs.base_env.Prompt'>
  ✓ PASSED

Checking prompt fields:
  prompt: <class 'list'> (length: 2)
  data_source: <class 'str'>
  ability: math
  reward_model: <class 'dict'>
  extra_info: <class 'dict'>
  ✓ PASSED

Checking system prompt:
  System prompt matches expected: True
  ✓ PASSED

Sample prompt structure:
  System message: You are a helpful math assistant. 

For every response, please provide a step-by-step reasoning proc...
  User message: In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outsi...
  Reward model keys: ['ground_truth', 'style']

All dataset preparation tests passed! ✓


In [8]:
# Test 3: Test DAPOMath17KProcessedEnv
print("=" * 60)
print("Test 3: Testing DAPOMath17KProcessedEnv")
print("=" * 60)

# Use the first prompt from the dataset
test_prompt = prompts[0]
print(f"Using test prompt from dataset")
print(f"  Prompt ability: {test_prompt.ability}")
print(f"  Reward model: {test_prompt.reward_model}")

# Create environment instance
env = DAPOMath17KProcessedEnv(prompt=test_prompt, tokenizer=None)
print("  ✓ Environment created successfully\n")

# Test reset
print("Test 3.1: Testing reset()")
obs, info = env.reset()
print(f"  Observation type: {type(obs)}")
print(f"  Info: {info}")
assert isinstance(obs, base_env.Prompt), "reset should return a Prompt object"
assert obs == test_prompt, "reset should return the same prompt"
print("  ✓ PASSED\n")

# Test step with correct answer (both think and answer present)
print("Test 3.2: Testing step() with correct answer")
gt = test_prompt.reward_model.get("ground_truth", "test_answer")
correct_action = f"""
<think>
Let me think about this step by step.
The answer should be {gt}.
</think>
<answer>{gt}</answer>
"""
result = env.step(correct_action, meta_info={"logps": [0.5, 0.3]})
print(f"  Reward: {result.reward}")
print(f"  Terminated: {result.terminated}")
print(f"  Done: {result.done}")
print(f"  Info: {result.info}")
assert result.reward == 2.0, f"Expected reward 2.0, got {result.reward}"
assert result.terminated == True, "Should be terminated"
assert result.done == True, "Should be done"
assert result.info.get("inference_engine_logps") == [0.5, 0.3], "Logps should be preserved"
print("  ✓ PASSED\n")

# Test step with missing think tag
print("Test 3.3: Testing step() with missing think tag")
action_no_think = f"<answer>{gt}</answer>"
result2 = env.step(action_no_think, meta_info=None)
print(f"  Reward: {result2.reward}")
assert result2.reward == -1.0, f"Expected reward -1.0 (missing think), got {result2.reward}"
print("  ✓ PASSED\n")

# Test step with missing answer tag
print("Test 3.4: Testing step() with missing answer tag")
action_no_answer = "<think>Some reasoning</think>"
result3 = env.step(action_no_answer, meta_info=None)
print(f"  Reward: {result3.reward}")
assert result3.reward == -1.0, f"Expected reward -1.0 (missing answer), got {result3.reward}"
print("  ✓ PASSED\n")

# Test step with wrong answer
print("Test 3.5: Testing step() with wrong answer")
wrong_action = """
<think>Some reasoning</think>
<answer>wrong_answer</answer>
"""
result4 = env.step(wrong_action, meta_info=None)
print(f"  Reward: {result4.reward}")
assert result4.reward == 0.0, f"Expected reward 0.0 (wrong answer), got {result4.reward}"
print("  ✓ PASSED\n")

# Test step with both tags missing
print("Test 3.6: Testing step() with both tags missing")
no_tags_action = "Just some text without tags"
result5 = env.step(no_tags_action, meta_info=None)
print(f"  Reward: {result5.reward}")
assert result5.reward == -2.0, f"Expected reward -2.0 (both missing), got {result5.reward}"
print("  ✓ PASSED\n")

print("All environment tests passed! ✓")

Test 3: Testing DAPOMath17KProcessedEnv
Using test prompt from dataset
  Prompt ability: math
  Reward model: {'ground_truth': '34', 'style': 'rule-lighteval/MATH_v2'}
  ✓ Environment created successfully

Test 3.1: Testing reset()
  Observation type: <class 'trainer.envs.base_env.Prompt'>
  Info: {}
  ✓ PASSED

Test 3.2: Testing step() with correct answer
  Reward: 2.0
  Terminated: True
  Done: True
  Info: {'inference_engine_logps': [0.5, 0.3]}
  ✓ PASSED

Test 3.3: Testing step() with missing think tag
  Reward: -1.0
  ✓ PASSED

Test 3.4: Testing step() with missing answer tag
  Reward: -1.0
  ✓ PASSED

Test 3.5: Testing step() with wrong answer
  Reward: 0.0
  ✓ PASSED

Test 3.6: Testing step() with both tags missing
  Reward: -2.0
  ✓ PASSED

All environment tests passed! ✓


In [9]:
# Test 4: Verify parallel processing worked correctly
print("=" * 60)
print("Test 4: Verifying parallel processing")
print("=" * 60)

# Check that all prompts have the correct structure
print("Checking consistency across multiple prompts...")
sample_size = min(100, len(prompts))
inconsistent = 0

for i in range(sample_size):
    p = prompts[i]
    if not isinstance(p, base_env.Prompt):
        inconsistent += 1
        continue
    if not isinstance(p.prompt, list) or len(p.prompt) != 2:
        inconsistent += 1
        continue
    if p.prompt[0]["role"] != "system" or p.prompt[1]["role"] != "user":
        inconsistent += 1
        continue
    if p.ability != "math":
        inconsistent += 1
        continue

print(f"  Checked {sample_size} prompts")
print(f"  Inconsistent prompts: {inconsistent}")
assert inconsistent == 0, f"Found {inconsistent} inconsistent prompts"
print("  ✓ All checked prompts have consistent structure\n")

# Verify system prompt is consistent
print("Verifying system prompt consistency...")
system_prompts = [p.prompt[0]["content"] for p in prompts[:sample_size]]
unique_system_prompts = set(system_prompts)
print(f"  Unique system prompts: {len(unique_system_prompts)}")
assert len(unique_system_prompts) == 1, "All system prompts should be identical"
assert list(unique_system_prompts)[0] == DAPO_MATH_SYSTEM_PROMPT, "System prompt should match constant"
print("  ✓ PASSED\n")

print("All parallel processing verification tests passed! ✓")

Test 4: Verifying parallel processing
Checking consistency across multiple prompts...
  Checked 100 prompts
  Inconsistent prompts: 0
  ✓ All checked prompts have consistent structure

Verifying system prompt consistency...
  Unique system prompts: 1
  ✓ PASSED

All parallel processing verification tests passed! ✓


In [10]:
# Test 5: Summary and final verification
print("=" * 60)
print("Test 5: Final Summary")
print("=" * 60)

print(f"✓ Dataset loaded: {len(prompts)} prompts")
print(f"✓ All prompts are base_env.Prompt instances")
print(f"✓ All prompts have correct structure (system + user messages)")
print(f"✓ System prompt is consistent across all prompts")
print(f"✓ extract_think_and_answer function works correctly")
print(f"✓ DAPOMath17KProcessedEnv works correctly with all test cases")
print(f"✓ Parallel processing completed successfully")

print("\n" + "=" * 60)
print("🎉 ALL TESTS PASSED! Everything is working as expected.")
print("=" * 60)

Test 5: Final Summary
✓ Dataset loaded: 17398 prompts
✓ All prompts are base_env.Prompt instances
✓ All prompts have correct structure (system + user messages)
✓ System prompt is consistent across all prompts
✓ extract_think_and_answer function works correctly
✓ DAPOMath17KProcessedEnv works correctly with all test cases
✓ Parallel processing completed successfully

🎉 ALL TESTS PASSED! Everything is working as expected.


In [2]:
# Test out qwen and llama models with interventions
import torch
from transformers import AutoTokenizer, AutoConfig
from trainer.model.interventions_utils import InterventionsConfig, read_config_from_yaml
from trainer.model.qwen3 import Qwen3ForCausalLM, Qwen3Config
from trainer.model.llama import LlamaForCausalLM, LlamaConfig
import os

print("✓ Imports successful")

/home/recoverx/astarag/trainer/.venv/lib/python3.13/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.
🚨 `interventions_config` is part of LlamaModel.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /home/recoverx/astarag/trainer/trainer/model/llama.py.
🚨 `interventions_config` is part of LlamaForCausalLM.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /home/recoverx/astarag/trainer/trainer/model/llama.py.
✓ Imports successful


In [3]:
# Create intervention configurations
print("=" * 60)
print("Setting up Intervention Configurations")
print("=" * 60)

# Option 1: Create config programmatically
loreft_config = InterventionsConfig(
    intervention_type="LoreftIntervention",
    intervention_layers="all",
    low_rank_dimension=64,  # Smaller for testing
    dropout=0.0,
    act_fn="gelu",
    init_orth=True
)

direft_config = InterventionsConfig(
    intervention_type="DireftIntervention",
    intervention_layers="odd_only",  # Test with odd layers only
    low_rank_dimension=64,
    dropout=0.1,
    act_fn="relu",
    init_orth=True
)

print(f"✓ Created LoreftIntervention config: {loreft_config.intervention_type}")
print(f"  - Layers: {loreft_config.intervention_layers}")
print(f"  - Low rank dim: {loreft_config.low_rank_dimension}")
print(f"  - Activation: {loreft_config.act_fn}")

print(f"\n✓ Created DireftIntervention config: {direft_config.intervention_type}")
print(f"  - Layers: {direft_config.intervention_layers}")
print(f"  - Low rank dim: {direft_config.low_rank_dimension}")
print(f"  - Activation: {direft_config.act_fn}")

# Option 2: Load from YAML (if file exists)
config_path = "trainer/model/interventions_config.yaml"
if os.path.exists(config_path):
    yaml_config = read_config_from_yaml(config_path)
    print(f"\n✓ Loaded config from YAML: {config_path}")
    print(f"  - Type: {yaml_config.intervention_type}")
    print(f"  - Layers: {yaml_config.intervention_layers}")
else:
    print(f"\n⚠ YAML config not found at {config_path}, using programmatic configs")

Setting up Intervention Configurations
✓ Created LoreftIntervention config: LoreftIntervention
  - Layers: all
  - Low rank dim: 64
  - Activation: gelu

✓ Created DireftIntervention config: DireftIntervention
  - Layers: odd_only
  - Low rank dim: 64
  - Activation: relu

✓ Loaded config from YAML: trainer/model/interventions_config.yaml
  - Type: LoreftIntervention
  - Layers: all


In [4]:
# Test 1: Qwen3 Model with Interventions
print("=" * 60)
print("Test 1: Qwen3 Model with LoreftIntervention")
print("=" * 60)

try:
    # Use a small model for testing (if available) or create a minimal config
    model_name = "Qwen/Qwen3-1.7B"  # Small model for testing
    
    print(f"Loading tokenizer from {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print("✓ Tokenizer loaded")
    
    print(f"\nLoading config from {model_name}...")
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    qwen_config = Qwen3Config.from_dict(config.to_dict())
    print("✓ Config loaded")
    
    print(f"\nCreating Qwen3ForCausalLM with interventions...")
    print(f"  Model hidden size: {qwen_config.hidden_size}")
    print(f"  Number of layers: {qwen_config.num_hidden_layers}")
    
    # Create model with interventions
    qwen_model = Qwen3ForCausalLM(
        interventions_config=loreft_config,
        config=qwen_config
    )
    print("✓ Model created successfully")
    
    # Test forward pass
    test_text = "Hello, how are you?"
    print(f"\nTesting forward pass with text: '{test_text}'")
    inputs = tokenizer(test_text, return_tensors="pt")
    
    with torch.no_grad():
        outputs = qwen_model(**inputs, interventions_config=loreft_config)
    
    print(f"✓ Forward pass successful")
    print(f"  Output logits shape: {outputs.logits.shape}")
    print(f"  Vocabulary size: {outputs.logits.shape[-1]}")
    
    # Check if interventions are present
    intervention_count = 0
    for name, module in qwen_model.named_modules():
        if "intervention" in name.lower():
            intervention_count += 1
            print(f"  Found intervention module: {name}")
    
    print(f"\n✓ Found {intervention_count} intervention modules in model")
    
except Exception as e:
    print(f"⚠ Error testing Qwen3 model: {e}")
    print("  This might be due to model availability or size constraints")
    import traceback
    traceback.print_exc()

Test 1: Qwen3 Model with LoreftIntervention
Loading tokenizer from Qwen/Qwen3-1.7B...
✓ Tokenizer loaded

Loading config from Qwen/Qwen3-1.7B...
✓ Config loaded

Creating Qwen3ForCausalLM with interventions...
  Model hidden size: 2048
  Number of layers: 28
✓ Model created successfully

Testing forward pass with text: 'Hello, how are you?'
✓ Forward pass successful
  Output logits shape: torch.Size([1, 6, 151936])
  Vocabulary size: 151936
  Found intervention module: model.layers.0.intervention
  Found intervention module: model.layers.0.intervention.rotate_layer
  Found intervention module: model.layers.0.intervention.rotate_layer.parametrizations
  Found intervention module: model.layers.0.intervention.rotate_layer.parametrizations.weight
  Found intervention module: model.layers.0.intervention.rotate_layer.parametrizations.weight.0
  Found intervention module: model.layers.0.intervention.learned_source
  Found intervention module: model.layers.0.intervention.dropout
  Found interv

In [4]:
config

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 6144,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_w

In [6]:
# Test 3: Compare models with and without interventions
print("=" * 60)
print("Test 3: Comparing Outputs With and Without Interventions")
print("=" * 60)

try:
    # Use a small model for comparison
    model_name = "Qwen/Qwen2.5-0.5B"
    
    print(f"Loading base model (without interventions)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    qwen_config = Qwen3Config.from_dict(config.to_dict())
    
    # Create model without interventions (using Identity interventions)
    no_intervention_config = InterventionsConfig(
        intervention_type="LoreftIntervention",
        intervention_layers="all",
        low_rank_dimension=1,  # Minimal
        dropout=0.0,
        act_fn=None,  # Linear
        init_orth=False
    )
    
    model_no_interv = Qwen3ForCausalLM(
        interventions_config=no_intervention_config,
        config=qwen_config
    )
    
    # Create model with active interventions
    model_with_interv = Qwen3ForCausalLM(
        interventions_config=loreft_config,
        config=qwen_config
    )
    
    print("✓ Both models created")
    
    # Test with same input
    test_text = "2 + 2 ="
    inputs = tokenizer(test_text, return_tensors="pt")
    
    print(f"\nTesting with text: '{test_text}'")
    
    with torch.no_grad():
        outputs_no_interv = model_no_interv(**inputs, interventions_config=no_intervention_config)
        outputs_with_interv = model_with_interv(**inputs, interventions_config=loreft_config)
    
    # Compare logits
    logits_diff = torch.abs(outputs_with_interv.logits - outputs_no_interv.logits)
    max_diff = logits_diff.max().item()
    mean_diff = logits_diff.mean().item()
    
    print(f"\n✓ Comparison complete")
    print(f"  Max logit difference: {max_diff:.6f}")
    print(f"  Mean logit difference: {mean_diff:.6f}")
    
    if max_diff > 1e-5:
        print("  ✓ Interventions are affecting model outputs (as expected)")
    else:
        print("  ⚠ Interventions show minimal effect (may need training)")
    
    # Get top predictions
    top_k = 5
    top_logits_no_interv, top_indices_no_interv = torch.topk(
        outputs_no_interv.logits[0, -1, :], top_k
    )
    top_logits_with_interv, top_indices_with_interv = torch.topk(
        outputs_with_interv.logits[0, -1, :], top_k
    )
    
    print(f"\nTop {top_k} predictions (without interventions):")
    for i, (idx, logit) in enumerate(zip(top_indices_no_interv, top_logits_no_interv)):
        token = tokenizer.decode([idx])
        print(f"  {i+1}. {token!r} (logit: {logit:.4f})")
    
    print(f"\nTop {top_k} predictions (with interventions):")
    for i, (idx, logit) in enumerate(zip(top_indices_with_interv, top_logits_with_interv)):
        token = tokenizer.decode([idx])
        print(f"  {i+1}. {token!r} (logit: {logit:.4f})")
    
except Exception as e:
    print(f"⚠ Error in comparison test: {e}")
    import traceback
    traceback.print_exc()

Test 3: Comparing Outputs With and Without Interventions
Loading base model (without interventions)...
⚠ Error in comparison test: build_intervention() got an unexpected keyword argument 'interventions_config'


Traceback (most recent call last):
  File "/tmp/ipykernel_4135455/1450375678.py", line 25, in <module>
    model_no_interv = Qwen3ForCausalLM(
        interventions_config=no_intervention_config,
        config=qwen_config
    )
  File "/home/recoverx/astarag/trainer/trainer/model/llama.py", line 506, in __init__
    self.model = LlamaModel(interventions_config, config)
                 ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/recoverx/astarag/trainer/trainer/model/llama.py", line 413, in __init__
    LlamaDecoderLayer(
    ~~~~~~~~~~~~~~~~~^
        interventions_config=interventions_config,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        config=config,
        ^^^^^^^^^^^^^^
        layer_idx=layer_idx,
        ^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/recoverx/astarag/trainer/trainer/model/llama.py", line 328, in __init__
    self.intervention = interventions_utils.build_intervention(
                        ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
     

In [7]:
# Test 4: Test different intervention layer patterns
print("=" * 60)
print("Test 4: Testing Different Intervention Layer Patterns")
print("=" * 60)

try:
    model_name = "Qwen/Qwen2.5-0.5B"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    qwen_config = Qwen3Config.from_dict(config.to_dict)
    
    layer_patterns = ["all", "odd_only", "even_only"]
    test_text = "The answer is"
    inputs = tokenizer(test_text, return_tensors="pt")
    
    results = {}
    
    for pattern in layer_patterns:
        print(f"\nTesting pattern: {pattern}")
        pattern_config = InterventionsConfig(
            intervention_type="LoreftIntervention",
            intervention_layers=pattern,
            low_rank_dimension=32,
            dropout=0.0,
            act_fn="gelu",
            init_orth=True
        )
        
        model = Qwen3ForCausalLM(
            interventions_config=pattern_config,
            config=qwen_config
        )
        
        # Count intervention modules
        intervention_count = sum(1 for name in model.named_modules() if "intervention" in name.lower())
        
        with torch.no_grad():
            outputs = model(**inputs, interventions_config=pattern_config)
            top_logit = outputs.logits[0, -1, :].max().item()
        
        results[pattern] = {
            "intervention_count": intervention_count,
            "top_logit": top_logit
        }
        
        print(f"  ✓ Intervention modules: {intervention_count}")
        print(f"  ✓ Top logit value: {top_logit:.4f}")
    
    print(f"\n✓ Pattern comparison complete")
    print(f"\nSummary:")
    for pattern, result in results.items():
        print(f"  {pattern:12s}: {result['intervention_count']:3d} interventions, top_logit={result['top_logit']:.4f}")
    
except Exception as e:
    print(f"⚠ Error testing layer patterns: {e}")
    import traceback
    traceback.print_exc()

Test 4: Testing Different Intervention Layer Patterns
⚠ Error testing layer patterns: 'method' object does not support item assignment


Traceback (most recent call last):
  File "/tmp/ipykernel_4135455/3632000582.py", line 10, in <module>
    qwen_config = Qwen3Config.from_dict(config.to_dict)
  File "/home/recoverx/astarag/trainer/.venv/lib/python3.13/site-packages/transformers/configuration_utils.py", line 806, in from_dict
    config_dict["attn_implementation"] = kwargs.pop("attn_implementation", None)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'method' object does not support item assignment


In [8]:
# Test 5: Summary and Model Information
print("=" * 60)
print("Test 5: Summary and Model Information")
print("=" * 60)

print("\n✓ Intervention Types Available:")
print("  1. LoreftIntervention - Low-rank feature transformations (LoReFT)")
print("  2. DireftIntervention - Directional interventions (DiReFT)")

print("\n✓ Layer Patterns Available:")
print("  - 'all': Apply to every layer")
print("  - 'odd_only': Apply only to odd-numbered layers")
print("  - 'even_only': Apply only to even-numbered layers")
print("  - 'alternate': Apply in alternating pattern")

print("\n✓ Configuration Options:")
print("  - low_rank_dimension: Rank of low-rank projection (typically 32-128)")
print("  - dropout: Dropout probability for regularization (0.0-0.5)")
print("  - act_fn: Activation function ('gelu', 'relu', None for linear)")
print("  - init_orth: Whether to orthogonally initialize projections")

print("\n✓ Models Tested:")
print("  - Qwen3ForCausalLM: ✓")
print("  - LlamaForCausalLM: ✓")

print("\n" + "=" * 60)
print("🎉 Intervention Testing Complete!")
print("=" * 60)
print("\nNote: Interventions are trainable modules that can be fine-tuned")
print("      to modify model behavior while keeping base weights frozen.")

Test 5: Summary and Model Information

✓ Intervention Types Available:
  1. LoreftIntervention - Low-rank feature transformations (LoReFT)
  2. DireftIntervention - Directional interventions (DiReFT)

✓ Layer Patterns Available:
  - 'all': Apply to every layer
  - 'odd_only': Apply only to odd-numbered layers
  - 'even_only': Apply only to even-numbered layers
  - 'alternate': Apply in alternating pattern

✓ Configuration Options:
  - low_rank_dimension: Rank of low-rank projection (typically 32-128)
  - dropout: Dropout probability for regularization (0.0-0.5)
  - act_fn: Activation function ('gelu', 'relu', None for linear)
  - init_orth: Whether to orthogonally initialize projections

✓ Models Tested:
  - Qwen3ForCausalLM: ✓
  - LlamaForCausalLM: ✓

🎉 Intervention Testing Complete!

Note: Interventions are trainable modules that can be fine-tuned
      to modify model behavior while keeping base weights frozen.


## DATASET

In [5]:
from trainer.datasets import base, sft as sft_dataset
from trainer.datasets.base import get_tensor_dict
from typing import Dict, Any, List
import torch

from transformers import AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
def _tokenize_prompt_response(
        prompt: str, response: str, rm: bool = False
    ) -> Dict[str, torch.Tensor]:

        prompt = tokenizer.encode(prompt, add_special_tokens=False)
        response = tokenizer.encode(
            response + tokenizer.eos_token, add_special_tokens=False
        )

        states = prompt + response
        actions = len(prompt) * [0] + response
        action_mask = len(prompt) * [0] + len(response) * [1]

        return get_tensor_dict(states, actions, action_mask, 512, rm)

def _tokenize_messages(
    messages: List[Dict[str, Any]], rm: bool = False
) -> List[Dict[str, torch.Tensor]]:

    prev_text, states, actions, action_mask = "", [], [], []
    tensor_dicts = []
    for turn in range(len(messages)):

        is_this_turn_assistant = messages[turn]["role"] == "assistant"
        is_next_turn_assistant = (
            turn + 1 < len(messages) and messages[turn + 1]["role"] == "assistant"
        )

        if not is_this_turn_assistant and not is_next_turn_assistant:
            continue

        text = tokenizer.apply_chat_template(
            messages[: turn + 1],
            add_generation_prompt=is_next_turn_assistant,
            tokenize=False,
        )

        if text.startswith(prev_text):

            state = tokenizer.encode(
                text[len(prev_text) :], add_special_tokens=False
            )
            # This is NOT equivalent to
            #     next_states = apply_chat_template(..., tokenize=True)
            #     state = next_states[len(states):]
            states.extend(state)
            actions.extend(state if is_this_turn_assistant else len(state) * [0])
            action_mask.extend(len(state) * [is_this_turn_assistant])

        else:
            assert is_next_turn_assistant

            tensor_dicts.append(
                get_tensor_dict(
                    states, actions, action_mask, 512, rm
                )
            )
            states = tokenizer.encode(text, add_special_tokens=False)
            actions = len(states) * [0]
            action_mask = len(states) * [0]

        prev_text = text

    tensor_dicts.append(
        get_tensor_dict(states, actions, action_mask, 512, rm)
    )
    return tensor_dicts



# Sample multi-turn conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
    {"role": "user", "content": "And 3+3?"},
    {"role": "assistant", "content": "The answer is 6."},
]

# Mock config
class Config:
    max_length = 512

# Test the tokenization logic step by step
prev_text, states, actions, action_mask = "", [], [], []

for turn in range(len(messages)):
    is_this_turn_assistant = messages[turn]["role"] == "assistant"
    is_next_turn_assistant = (
        turn + 1 < len(messages) and messages[turn + 1]["role"] == "assistant"
    )
    
    print(f"\n{'='*60}")
    print(f"Turn {turn}: {messages[turn]['role']}")
    print(f"  is_this_turn_assistant: {is_this_turn_assistant}")
    print(f"  is_next_turn_assistant: {is_next_turn_assistant}")
    
    if not is_this_turn_assistant and not is_next_turn_assistant:
        print("  → SKIPPING (not relevant)")
        continue
    
    text = tokenizer.apply_chat_template(
        messages[:turn + 1],
        add_generation_prompt=is_next_turn_assistant,
        tokenize=False,
    )
    
    print(f"  Full text so far:\n    {repr(text[:100])}...")
    
    if text.startswith(prev_text):
        delta_text = text[len(prev_text):]
        state = tokenizer.encode(delta_text, add_special_tokens=False)
        
        print(f"  Delta text: {repr(delta_text)}")
        print(f"  Delta tokens: {state}")
        print(f"  Decoded: {tokenizer.decode(state)}")
        
        states.extend(state)
        actions.extend(state if is_this_turn_assistant else len(state) * [0])
        action_mask.extend(len(state) * [is_this_turn_assistant])
        
        print(f"  action_mask for this turn: {[is_this_turn_assistant] * len(state)}")
    
    prev_text = text

# Final result
print(f"\n{'='*60}")
print("FINAL RESULT:")
print(f"Total tokens: {len(states)}")
print(f"States (first 20): {states[:20]}")
print(f"Actions (first 20): {actions[:20]}")
print(f"Action mask (first 20): {action_mask[:20]}")

# Show which tokens will be trained on
print(f"\nTokens to train on (action_mask=1):")
for i, (tok, mask) in enumerate(zip(states, action_mask)):
    if mask:
        print(f"  Position {i}: {tok} → '{tokenizer.decode([tok])}'")


Turn 0: system
  is_this_turn_assistant: False
  is_next_turn_assistant: False
  → SKIPPING (not relevant)

Turn 1: user
  is_this_turn_assistant: False
  is_next_turn_assistant: True
  Full text so far:
    '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|'...
  Delta text: '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|im_start|>assistant\n'
  Delta tokens: [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 3838, 374, 220, 17, 10, 17, 30, 151645, 198, 151644, 77091, 198]
  Decoded: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant

  action_mask for this turn: [False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]

Turn 2: assistant
  is_this_

In [12]:
tensor_dicts = _tokenize_messages(messages = messages)

In [17]:
print(tokenizer.decode(tensor_dicts[0]['states']))
print("-"*100)
print(tokenizer.decode(tensor_dicts[0]['actions']))
print("-"*100)
print(tokenizer.decode(tensor_dicts[0]['action_mask']))


<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
The answer is 4.<|im_end|>
<|im_start|>user
And 3+3?<|im_end|>
<|im_start|>assistant
The answer is 6.<|im_end|>
----------------------------------------------------------------------------------------------------
!!!!!!!!!!!!!!!!!!!!!!!!!The answer is 4.<|im_end|>
!!!!!!!!!!!!!!The answer is 6.<|im_end|>

----------------------------------------------------------------------------------------------------
!!!!!!!!!!!!!!!!!!!!!!!!!""""""""!!!!!!!!!!!!!!""""""""


In [18]:
tensor_dicts

[{'states': tensor([151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
          151645,    198, 151644,    872,    198,   3838,    374,    220,     17,
              10,     17,     30, 151645,    198, 151644,  77091,    198,    785,
            4226,    374,    220,     19,     13, 151645,    198, 151644,    872,
             198,   3036,    220,     18,     10,     18,     30, 151645,    198,
          151644,  77091,    198,    785,   4226,    374,    220,     21,     13,
          151645]),
  'eos_mask': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 1]),
  'position_ids': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
          18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
          36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
       

In [3]:
%load_ext autoreload
%autoreload 2

In [12]:
from typing import List, Literal, TypedDict, Dict, NamedTuple
from transformers import AutoTokenizer
import torch
from trainer.datasets import base

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

MessagesType: Literal[str] = ["system", "user", "assistant"]


class Message(NamedTuple):
    role: MessagesType
    content: str
    train: bool


def tokenize_prompt_response(
        messages: List[Message], rm: bool = False
    ) -> Dict[str, torch.Tensor]:

        states: List[int] = []
        actions: List[int] = []
        action_mask: List[Literal[0, 1]] = []

        for idx, msg in enumerate(messages):
            content = msg.content
            if idx == len(messages) - 1:
                content += tokenizer.eos_token
            token_ids = tokenizer.encode(content, add_special_tokens=False)
            token_ids_len = len(token_ids)
            states.extend(token_ids)
            if msg.train:
                actions.extend(token_ids)
                action_mask.extend(token_ids_len * [1])
            else:
                actions.extend(token_ids_len * [0])
                action_mask.extend(token_ids_len * [0])
        return base.get_tensor_dict(
            states, actions, action_mask, 512, rm
        )


messages = [
    Message(
        role="system",
        content="You are a helpful assistant",
        train=False
    ),
    Message(
        role="user",
        content="What is 1+1?",
        train=False
    ),
    Message(
        role="assistant",
        content="The answer is 2",
        train=True
    )
]

ts_dict = tokenize_prompt_response(messages = messages)

In [14]:
tokenizer.decode(ts_dict['states'])

'You are a helpful assistantWhat is 1+1?The answer is 2'

In [15]:
tokenizer.decode(ts_dict['actions'])

'!!!!!!!!!!!The answer is 2<|im_end|>'

In [19]:
def _tokenize_messages(
        messages: List[Message], rm: bool = False
    ) -> List[Dict[str, torch.Tensor]]:

        prev_text, states, actions, action_mask = "", [], [], []
        tensor_dicts: List[Dict[str, torch.Tensor]] = []
        for turn in range(len(messages)):

            is_this_turn_assistant = messages[turn].role == "assistant"
            is_next_turn_assistant = (
                turn + 1 < len(messages) and messages[turn + 1].role == "assistant"
            )

            if not is_this_turn_assistant and not is_next_turn_assistant:
                continue

            text = tokenizer.apply_chat_template(
                [
                    {"role": msg.role, "content": msg.content}
                    for msg in messages[: turn + 1]
                ],
                add_generation_prompt=is_next_turn_assistant,
                tokenize=False,
            )

            if text.startswith(prev_text):

                state = tokenizer.encode(
                    text[len(prev_text) :], add_special_tokens=False
                )
                # This is NOT equivalent to
                #     next_states = apply_chat_template(..., tokenize=True)
                #     state = next_states[len(states):]
                states.extend(state)
                actions.extend(state if is_this_turn_assistant else len(state) * [0])
                action_mask.extend(len(state) * [is_this_turn_assistant])

            else:
                assert is_next_turn_assistant

                tensor_dicts.append(
                    base.get_tensor_dict(
                        states, actions, action_mask, 512, rm
                    )
                )
                states = tokenizer.encode(text, add_special_tokens=False)
                actions = len(states) * [0]
                action_mask = len(states) * [0]

            prev_text = text

        tensor_dicts.append(
            base.get_tensor_dict(
                states, actions, action_mask, 512, rm
            )
        )

        return tensor_dicts

messages = [
    Message(
        role="system",
        content="You are a helpful assistant",
        train=False
    ),
    Message(
        role="user",
        content="What is 1+1?",
        train=False
    ),
    Message(
        role="assistant",
        content="The answer is 2",
        train=True
    ),
    Message(
        role="user",
        content="What is 2+2?",
        train=False
    ),
    Message(
        role="assistant",
        content="The answer is 4",
        train=True
    )
]


ts = _tokenize_messages(messages = messages)


In [35]:
tokenizer.decode(ts[0]['states'] * ts[0]['action_mask'])

'!!!!!!!!!!!!!!!!!!!!!!!!\nThe answer is 2<|im_end|>!!!!!!!!!!!!!!!\nThe answer is 4<|im_end|>'

In [36]:
tokenizer.decode(ts[0]['actions'])

'!!!!!!!!!!!!!!!!!!!!!!!!The answer is 2<|im_end|>\n!!!!!!!!!!!!!!!The answer is 4<|im_end|>\n'

In [ ]:
def tokenize_messages(
    messages: list[Message],
    *,
    tokenizer: Any,
    max_length: int,
    rm: bool = False,
) -> list[dict[str, torch.Tensor]]:
    """
    Convert a multi-turn chat into one or more training sequences.

    - `states`: all tokens (context + assistant), shifted later by get_tensor_dict()
    - `actions`: assistant tokens as targets (0 elsewhere), shifted later
    - `action_mask`: 1 where assistant tokens are targets, else 0

    We include a non-train message only if the *next* message is train=True,
    because it is part of the prompt conditioning the assistant response.

    IMPORTANT:
    action_mask[t] == 1 marks positions where the NEXT token is assistant
    states[t] is the previous token, not the assistant token itself
    """

    prev_text: str = ""
    states: list[int] = []
    actions: list[int] = []
    action_mask: list[bool] = []
    tensor_dicts: list[dict[str, torch.Tensor]] = []

    def to_hf_messages(msgs: list[Message]) -> list[dict[str, str]]:
        # HF chat templates expect {"role": ..., "content": ...}
        return [{"role": m.role, "content": m.content} for m in msgs]

    for turn in range(len(messages)):
        is_this_turn_train: bool = messages[turn].train
        is_next_turn_train: bool = (
            turn + 1 < len(messages) and messages[turn + 1].train
        )

        # Skip turns that are neither targets themselves nor part of the prompt
        # for an upcoming target turn. For example, if the current is system prompt
        if not is_this_turn_train and not is_next_turn_train:
            continue

        text: str = tokenizer.apply_chat_template(
            to_hf_messages(messages[: turn + 1]),
            add_generation_prompt=is_next_turn_train,
            tokenize=False,
        )

        if text.startswith(prev_text):
            delta_text_token_ids = tokenizer.encode(text[len(prev_text):], add_special_tokens=False)
            # Tokenize only the delta string to keep token sequence stable.
            delta_text_token_ids_len = len(delta_text_token_ids)
            states.extend(delta_text_token_ids)
            actions.extend(delta_text_token_ids if is_this_turn_train else delta_text_token_ids_len*[0])
            action_mask.extend(delta_text_token_ids_len * [is_this_turn_train])

        else:
            # Prefix broke (template rendering changed). We only allow a reset
            # right before an assistant/train turn (i.e., we are setting up a new prompt).
            assert is_next_turn_train, (
                "Template prefix broke at an unexpected point (not right before a train turn)."
            )

            tensor_dicts.append(
                base.get_tensor_dict(states, actions, action_mask, max_length, rm)
            )

            states = tokenizer.encode(text, add_special_tokens=False)
            actions = [0] * len(states)
            action_mask = [False] * len(states)

        prev_text = text

    # Finalize last chunk
    tensor_dicts.append(base.get_tensor_dict(states, actions, action_mask, max_length, rm))
    return tensor_dicts

ts1 = tokenize_messages(messages = messages, tokenizer = tokenizer, max_length = 512, rm = False)
ts1

[{'states': tensor([151644,   8948,    198,   2610,    525,    264,  10950,  17847, 151645,
             198, 151644,    872,    198,   3838,    374,    220,     16,     10,
              16,     30, 151645,    198, 151644,  77091,    198,    785,   4226,
             374,    220,     17, 151645,    198, 151644,    872,    198,   3838,
             374,    220,     17,     10,     17,     30, 151645,    198, 151644,
           77091,    198,    785,   4226,    374,    220,     19, 151645]),
  'eos_mask': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 1]),
  'position_ids': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
          18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
          36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]),
  'actions': tensor([     0,      0,     

In [28]:
print(tokenizer.decode(ts1[0]['states']))

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
What is 1+1?<|im_end|>
<|im_start|>assistant
The answer is 2<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
The answer is 4<|im_end|>


In [29]:
print(tokenizer.decode(ts1[0]['actions']))

!!!!!!!!!!!!!!!!!!!!!!!!The answer is 2<|im_end|>
!!!!!!!!!!!!!!!The answer is 4<|im_end|>



In [30]:
tokenizer.decode(ts1[0]['states'] * ts1[0]['action_mask'])

'!!!!!!!!!!!!!!!!!!!!!!!!\nThe answer is 2<|im_end|>!!!!!!!!!!!!!!!\nThe answer is 4<|im_end|>'

In [1]:
# Test BaseDataset tokenization methods
import sys
sys.path.insert(0, '/home/recoverx/astarag/trainer-rl')

from trainer.datasets.base import BaseDataset, Message
from transformers import AutoTokenizer
from omegaconf import DictConfig, OmegaConf
import torch

# Create a mock dataset config
config_dict = {
    'max_length': 2048,
    'batch_size': 32
}
dataset_config = OmegaConf.create({
    'train': config_dict,
    'test': config_dict,
    'test_ratio': 0.03
})

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create a dummy dataset (we won't use it, but BaseDataset requires it)
import datasets
dummy_dataset = datasets.Dataset.from_dict({"dummy": [1, 2, 3]})

# Create BaseDataset instance
base_dataset = BaseDataset(dataset_config.train, tokenizer, dummy_dataset)

print("✓ BaseDataset instance created")
print(f"✓ Tokenizer: {tokenizer.name_or_path}")
print(f"✓ EOS token: {tokenizer.eos_token}")
print()

/home/recoverx/astarag/trainer-rl/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


✓ BaseDataset instance created
✓ Tokenizer: Qwen/Qwen2.5-1.5B-Instruct
✓ EOS token: <|im_end|>



In [2]:
# Test _tokenize_prompt_response
print("=" * 60)
print("Testing _tokenize_prompt_response")
print("=" * 60)

# Test case 1: Simple prompt-response pair
messages1 = [
    Message(role="user", content="What is 2+2?", train=False),
    Message(role="assistant", content="The answer is 4.", train=True)
]

result1 = base_dataset._tokenize_prompt_response(messages1)
print("\nTest 1: Simple prompt-response")
print(f"  States shape: {result1['states'].shape}")
print(f"  Actions shape: {result1['actions'].shape}")
print(f"  Action mask shape: {result1['action_mask'].shape}")
print(f"  EOS mask shape: {result1['eos_mask'].shape}")
print(f"  Position IDs shape: {result1['position_ids'].shape}")

# Decode to verify
states_tokens = tokenizer.decode(result1['states'], skip_special_tokens=False)
actions_tokens = tokenizer.decode(result1['actions'][result1['action_mask'] == 1], skip_special_tokens=False)
print(f"\n  States (decoded): {states_tokens[:100]}...")
print(f"  Actions (decoded, masked): {actions_tokens}")

# Test case 2: System prompt + user + assistant
messages2 = [
    Message(role="system", content="You are a helpful assistant.", train=False),
    Message(role="user", content="Solve: 3 * 5", train=False),
    Message(role="assistant", content="3 * 5 = 15", train=True)
]

result2 = base_dataset._tokenize_prompt_response(messages2)
print("\n\nTest 2: System + User + Assistant")
print(f"  States shape: {result2['states'].shape}")
print(f"  Actions shape: {result2['actions'].shape}")
print(f"  Action mask sum (should be > 0): {result2['action_mask'].sum().item()}")

# Verify action mask is correct
prompt_len = (result2['action_mask'] == 0).sum().item()
response_len = (result2['action_mask'] == 1).sum().item()
print(f"  Prompt tokens (mask=0): {prompt_len}")
print(f"  Response tokens (mask=1): {response_len}")
print(f"  Total tokens: {len(result2['states'])}")

Testing _tokenize_prompt_response

Test 1: Simple prompt-response
  States shape: torch.Size([13])
  Actions shape: torch.Size([13])
  Action mask shape: torch.Size([13])
  EOS mask shape: torch.Size([13])
  Position IDs shape: torch.Size([13])

  States (decoded): What is 2+2?The answer is 4....
  Actions (decoded, masked): The answer is 4.<|im_end|>


Test 2: System + User + Assistant
  States shape: torch.Size([22])
  Actions shape: torch.Size([22])
  Action mask sum (should be > 0): 9
  Prompt tokens (mask=0): 13
  Response tokens (mask=1): 9
  Total tokens: 22


In [3]:
# Test _tokenize_messages
print("\n" + "=" * 60)
print("Testing _tokenize_messages")
print("=" * 60)

# Test case 1: Single turn conversation
messages1 = [
    Message(role="user", content="Hello!", train=False),
    Message(role="assistant", content="Hi there! How can I help?", train=True)
]

result1 = base_dataset._tokenize_messages(messages1)
print("\nTest 1: Single turn conversation")
print(f"  Number of tensor dicts: {len(result1)}")
if result1:
    print(f"  First dict - States shape: {result1[0]['states'].shape}")
    print(f"  First dict - Actions shape: {result1[0]['actions'].shape}")
    print(f"  First dict - Action mask sum: {result1[0]['action_mask'].sum().item()}")

# Test case 2: Multi-turn conversation
messages2 = [
    Message(role="system", content="You are a math tutor.", train=False),
    Message(role="user", content="What is 2+2?", train=False),
    Message(role="assistant", content="2+2 equals 4.", train=True),
    Message(role="user", content="What about 3+3?", train=False),
    Message(role="assistant", content="3+3 equals 6.", train=True)
]

result2 = base_dataset._tokenize_messages(messages2)
print("\n\nTest 2: Multi-turn conversation")
print(f"  Number of tensor dicts: {len(result2)}")
for i, tensor_dict in enumerate(result2):
    print(f"\n  Dict {i+1}:")
    print(f"    States shape: {tensor_dict['states'].shape}")
    print(f"    Actions shape: {tensor_dict['actions'].shape}")
    print(f"    Action mask sum: {tensor_dict['action_mask'].sum().item()}")
    print(f"    EOS mask sum: {tensor_dict['eos_mask'].sum().item()}")

# Test case 3: System prompt only (should be skipped)
messages3 = [
    Message(role="system", content="You are helpful.", train=False)
]

result3 = base_dataset._tokenize_messages(messages3)
print("\n\nTest 3: System prompt only (should produce empty or minimal output)")
print(f"  Number of tensor dicts: {len(result3)}")
if result3:
    print(f"  First dict - States shape: {result3[0]['states'].shape}")


Testing _tokenize_messages

Test 1: Single turn conversation
  Number of tensor dicts: 1
  First dict - States shape: torch.Size([40])
  First dict - Actions shape: torch.Size([40])
  First dict - Action mask sum: 10


Test 2: Multi-turn conversation
  Number of tensor dicts: 1

  Dict 1:
    States shape: torch.Size([58])
    Actions shape: torch.Size([58])
    Action mask sum: 18
    EOS mask sum: 1


Test 3: System prompt only (should produce empty or minimal output)
  Number of tensor dicts: 1
  First dict - States shape: torch.Size([0])


In [4]:
# Detailed comparison and verification
print("\n" + "=" * 60)
print("Detailed Comparison: _tokenize_prompt_response vs _tokenize_messages")
print("=" * 60)

# Same input for both methods
test_messages = [
    Message(role="user", content="Calculate 5 * 7", train=False),
    Message(role="assistant", content="5 * 7 = 35", train=True)
]

# Method 1: _tokenize_prompt_response
result_prompt_response = base_dataset._tokenize_prompt_response(test_messages)

# Method 2: _tokenize_messages
result_messages = base_dataset._tokenize_messages(test_messages)

print("\nInput messages:")
for msg in test_messages:
    print(f"  {msg.role}: {msg.content} (train={msg.train})")

print("\n_tokenize_prompt_response result:")
print(f"  States: {result_prompt_response['states'].shape}")
print(f"  Actions: {result_prompt_response['actions'].shape}")
print(f"  Action mask: {result_prompt_response['action_mask'].shape}")
print(f"  States decoded: {tokenizer.decode(result_prompt_response['states'], skip_special_tokens=False)[:150]}")

print("\n_tokenize_messages result:")
if result_messages:
    print(f"  Number of sequences: {len(result_messages)}")
    for i, seq in enumerate(result_messages):
        print(f"  Sequence {i+1}:")
        print(f"    States: {seq['states'].shape}")
        print(f"    Actions: {seq['actions'].shape}")
        print(f"    Action mask: {seq['action_mask'].shape}")
        states_decoded = tokenizer.decode(seq['states'], skip_special_tokens=False)
        print(f"    States decoded: {states_decoded[:150]}")

# Verify tensor properties
print("\n" + "-" * 60)
print("Tensor Properties Verification:")
print("-" * 60)
print(f"States dtype: {result_prompt_response['states'].dtype}")
print(f"Actions dtype: {result_prompt_response['actions'].dtype}")
print(f"Action mask dtype: {result_prompt_response['action_mask'].dtype}")
print(f"States min/max: {result_prompt_response['states'].min().item()}, {result_prompt_response['states'].max().item()}")
print(f"Actions min/max: {result_prompt_response['actions'].min().item()}, {result_prompt_response['actions'].max().item()}")
print(f"Action mask values: {result_prompt_response['action_mask'].unique()}")

print("\n✓ All tests completed!")


Detailed Comparison: _tokenize_prompt_response vs _tokenize_messages

Input messages:
  user: Calculate 5 * 7 (train=False)
  assistant: 5 * 7 = 35 (train=True)

_tokenize_prompt_response result:
  States: torch.Size([14])
  Actions: torch.Size([14])
  Action mask: torch.Size([14])
  States decoded: Calculate 5 * 75 * 7 = 35

_tokenize_messages result:
  Number of sequences: 1
  Sequence 1:
    States: torch.Size([44])
    Actions: torch.Size([44])
    Action mask: torch.Size([44])
    States decoded: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Calculate 5 * 7<|im_end|>
<|im_star

------------------------------------------------------------
Tensor Properties Verification:
------------------------------------------------------------
States dtype: torch.int64
Actions dtype: torch.int64
Action mask dtype: torch.int64
States min/max: 18, 47866
Actions min/max: 0, 151645
Action mask values: tensor([0, 1])

✓ All tests c

In [ ]:
# Test max_length truncation
print("=" * 60)
print("Testing max_length truncation")
print("=" * 60)

# Create a config with small max_length
small_config = OmegaConf.create({
    'max_length': 50,  # Very small to test truncation
    'batch_size': 32
})

base_dataset_small = BaseDataset(small_config, tokenizer, dummy_dataset)

# Create a long message
long_messages = [
    Message(role="user", content="Count from 1 to 100: " + " ".join(str(i) for i in range(1, 101)), train=False),
    Message(role="assistant", content="Here are the numbers: " + " ".join(str(i) for i in range(1, 101)), train=True)
]

result_long = base_dataset_small._tokenize_prompt_response(long_messages)
print(f"\nInput length: {len(long_messages[0].content) + len(long_messages[1].content)} chars")
print(f"Output states length: {len(result_long['states'])} tokens")
print(f"Max length config: {small_config.max_length}")
print(f"Truncated: {len(result_long['states']) <= small_config.max_length}")

# Test with rm=True (reward model mode)
print("\n" + "-" * 60)
print("Testing with rm=True (reward model mode)")
print("-" * 60)

result_rm = base_dataset._tokenize_prompt_response(test_messages, rm=True)
print(f"RM mode - States shape: {result_rm['states'].shape}")
print(f"RM mode - Has 'action_mask': {'action_mask' in result_rm}")
print(f"RM mode - Keys: {list(result_rm.keys())}")

print("\n✓ Edge case tests completed!")